---
title: Volume spike ratio
execute:
  enabled: true
---

`q.indicator.volume_spike_ratio` compares current volume with trailing average volume and combines it with the slope of the close-price SMA.

`vsr_spike_type` uses 0 for no classified spike, 1 for continuation, and 2 for reversal. `vsr_trending` uses 0 for no trend, 1 for uptrend, and 2 for downtrend. Prices and volume must describe one instrument and have identical indexes.

In [1]:
import pandas as pd

import qrt as q

data = q.data.datasets.load("aapl")
end = data.index.max()
data = data.loc[end - pd.DateOffset(years=5) :].copy()
result = q.indicator.volume_spike_ratio(data["close"], data["volume"])
events = data[["close", "volume"]].join(result)
events.loc[events["vsr_spike_type"].ne(0)].tail(10)

,close,volume,vsr,vsr_spike_type,vsr_trending
datetime,,,,,
2025-09-22,255.357559,105517400,1.791227,1,1
2025-10-20,261.500183,90483000,1.961588,1,1
2025-10-30,270.634338,69886500,1.519881,1,1
2026-01-16,255.056137,72142800,1.617250,1,2
2026-01-20,246.242508,80267500,1.743424,1,2
2026-03-20,247.761734,88331100,2.128798,1,2
2026-04-30,271.100250,91848200,2.117589,1,1
2026-05-01,279.882141,79915400,1.744678,1,1
2026-06-25,275.149994,107013700,1.900583,1,2


In [2]:
import plotly.graph_objects as go

figure = go.Figure()
figure.add_trace(go.Scatter(x=events.index, y=events["vsr"], name="VSR", line={"color": "#2457a7"}))
for code, name, color in [(1, "Continuation", "#16856b"), (2, "Reversal", "#d04a35")]:
    selected = events.loc[events["vsr_spike_type"].eq(code)]
    figure.add_trace(
        go.Scatter(
            x=selected.index,
            y=selected["vsr"],
            name=name,
            mode="markers",
            marker={"color": color, "size": 8},
        )
    )
figure.update_layout(title="AAPL volume spike ratio classifications", yaxis_title="Relative volume", hovermode="x unified")
figure.show(renderer="notebook_connected")